# Qwen-Image-Edit-2509 OCR Experiment

This notebook demonstrates an experiment where we use Qwen-Image-Edit-2509 to stylize text in an image, then use an OCR model to detect and extract the text from the modified image, and finally compare the detected text with the original text.

In [ ]:
import os
import torch
from PIL import Image
from diffusers import QwenImageEditPlusPipeline

# Create results directory if it doesn't exist
os.makedirs("results", exist_ok=True)

## Load Model and Input Image

First, we'll load the Qwen-Image-Edit-2509 pipeline and move it to the GPU. We'll also load the input image needed for the experiment.

In [ ]:
# Load pipeline
pipeline = QwenImageEditPlusPipeline.from_pretrained("Qwen/Qwen-Image-Edit-2509", torch_dtype=torch.bfloat16)
pipeline.to('cuda')
pipeline.set_progress_bar_config(disable=True)
print("Pipeline loaded on CUDA")

# Load input image
# You need to provide input.png in the working directory
# Or modify the path below to point to your image
input_image = Image.open("input.png")
print("Input image loaded")

## Stylize Text in Image

We'll use the Qwen-Image-Edit-2509 model to transform all English text in the image to be stylized and fully capitalized.

In [ ]:
# Define the prompt for stylizing text
prompt = "Make all English text in this image fully capitalized"

# Prepare inputs
inputs = {
    "image": input_image,
    "prompt": prompt,
    "generator": torch.manual_seed(0),
    "true_cfg_scale": 4.0,
    "negative_prompt": " ",
    "num_inference_steps": 40,
    "guidance_scale": 1.0,
    "num_images_per_prompt": 1,
}

# Run inference
with torch.inference_mode():
    output = pipeline(**inputs)

# Save the stylized image
stylized_image = output.images[0]
stylized_image_path = "results/stylized_image.png"
stylized_image.save(stylized_image_path)
print(f"Stylized image saved at {os.path.abspath(stylized_image_path)}")

## Extract Text with OCR

Now we'll use a state-of-the-art OCR model to extract text from the stylized image. For this experiment, we'll use EasyOCR which is a popular choice for text detection.

In [ ]:
# Install EasyOCR if not already installed
!pip install easyocr

import easyocr

# Initialize the OCR reader
reader = easyocr.Reader(['en'])
print("OCR reader initialized")

# Extract text from the stylized image
results = reader.readtext(stylized_image_path)
detected_text = " ".join([text for (bbox, text, confidence) in results])

# Save detected text to results.txt
results_file_path = "results/results.txt"
with open(results_file_path, "w") as f:
    f.write(detected_text)
print(f"Detected text saved to {os.path.abspath(results_file_path)}")
print("Detected text:")
print(detected_text)

## Compare Text and Calculate Match Percentage

Finally, we'll compare the detected text with the original text stored in control.txt and calculate the percentage match. We need to ensure control.txt exists in the working directory with the original text content.

In [ ]:
# Load original text from control.txt
try:
    with open("control.txt", "r") as f:
        original_text = f.read().strip()
    print("Original text loaded from control.txt")
except FileNotFoundError:
    print("control.txt not found. Please create a control.txt file with the original text content.")
    # For demonstration, we'll use a placeholder
    original_text = "This is placeholder text for the original content"

# Calculate percentage match
def calculate_match_percentage(original, detected):
    # Simple character-level comparison
    original_chars = set(original.lower())
    detected_chars = set(detected.lower())
    
    if len(original_chars) == 0:
        return 100.0 if len(detected_chars) == 0 else 0.0
    
    common_chars = original_chars.intersection(detected_chars)
    match_percentage = (len(common_chars) / len(original_chars)) * 100
    return match_percentage

# Calculate and display match percentage
match_percentage = calculate_match_percentage(original_text, detected_text)
print(f"\nText match percentage: {match_percentage:.2f}%")

# Detailed comparison
print("\nDetailed comparison:")
print(f"Original text: {original_text}")
print(f"Detected text: {detected_text}")